In [84]:
import pandas as pd 
import os

In [85]:
Solcast_Path = 'Data/Solcast/Solcast_2022-2023_unformatted.csv'
year = 2022
# Checking if the necessary files exist for formatting.
if not os.path.exists(Solcast_Path):
    print(f"Warning: This script will not run without the original Solcast data in {Solcast_Path}`.\n",
          'Please download the data from `solcast.com` at 5 minute cadence for all of 2023.')

In [86]:
df = pd.read_csv('Data/Solcast/Solcast_2022-2023_unformatted.csv')
stats = pd.read_csv('Data/Analysis/Data_Tables/csv/statistics.csv')
consts = pd.read_csv('Data/Analysis/Const/Const.csv', index_col=0)

# Making the date and time readable by Python.
format_date = '%Y-%m-%dT%H:%M:%S%z'
df['DateTime'] = pd.to_datetime(df['period_end'],format=format_date)

# Removing the UTC formatting of the date and time.
df['DateTime'] = df['DateTime'].dt.tz_localize(None)

# Setting the index to DateTime.
df.index = df.DateTime

In [87]:
# Filter so the data are of the same length as the DEOP data.
df = df.loc[(df.index >= f'{year}-01-01 00:05')
                     & (df.index <= f'{year}-12-31 23:55')]

In [88]:
df['P_MaxSolar'] = df['gti']*(consts.loc['A_pvUnit']*consts.loc['N_pv']).values[0]/1e3

In [89]:
df['expected'] = stats.loc['eta_Solar',f'{year}']*df['P_MaxSolar']

In [90]:
df.to_csv(f'Data/DEOP/{year}_Expected.csv', index=False, columns=['DateTime', 'expected'])

In [91]:
df.to_csv(f'Data/Solcast/Solcast_{year}.csv',index=False,
          columns=['air_temp', 'gti','surface_pressure', 
                   'snow_depth','cloud_opacity','ghi','clearsky_gti','wind_speed_100m','weather_type', 'DateTime'])